# 2. Evaluate - original CRM data (`data/crm/`)

Same step like `python -m src.evaluate --output reports/crm/final`, using function in `src/evaluate.py`.
It load the model from `1_train.ipynb`, nothing in here can changing the selected model, threshold or calibration.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "train.py").is_file())
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display
from src import evaluate, train
from src.data.crm import CRM_DIR

## Settings

Need to be same setting as in `1_train.ipynb`.

In [ ]:
DATA_DIR = CRM_DIR
QUICK = False
OUTPUT = train.default_output(DATA_DIR, QUICK)
print("Data:  ", DATA_DIR)
print("Output:", OUTPUT)

## 1. Load saved model and rebuild same split

In [ ]:
ev = evaluate.start(OUTPUT)
print("Data:", ev.out.source_data())
print("Selected model:", ev.trained.selected, "| mode:", ev.manifest["mode"])
print("Test deals:", len(ev.split.test))

## 2. Test metrics

Each models use it's own validation threshold. The Dummy one is just for show what you getting by predict the majority class.

In [ ]:
COLUMNS = ["model", "roc_auc", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "brier", "tn", "fp", "fn", "tp"]
evaluate.score_test(ev)[COLUMNS].round(4)

### Calibration of selected model

Calibration only change the probability scale (Brier), the ranking (ROC-AUC) is stay same.

In [ ]:
pd.read_csv(ev.out.metrics / "calibration_test.csv")[["variant", "roc_auc", "brier", "log_loss", "accuracy", "f1", "threshold"]].round(4)

## 3. Figures

In [ ]:
evaluate.make_figures(ev)
display(Image(filename=str(ev.out.figures / "roc_curves.png"), width=600))
display(Image(filename=str(ev.out.figures / "reliability.png"), width=500))
display(Image(filename=str(ev.out.figures / f"confusion_{ev.trained.selected}.png"), width=350))

## 4. Explanations

Permutation importance = how much test ROC-AUC is drop when we shuffle one input. TreeSHAP is only for raw Random Forest.
Both of them describe how model behave, not the cause.

In [ ]:
evaluate.explain_models(ev)
display(pd.read_csv(ev.out.explain / "permutation_importance_test.csv").head(10).round(4))
pd.read_csv(ev.out.explain / "rf_shap_global.csv").head(10).round(4)

## 5. Audits

The close-value model is invalid on purpose, it show how a outcome field can faking a almost perfect score.

In [ ]:
evaluate.run_audits(ev)
display(pd.read_csv(ev.out.checks / "leakage_audit.csv")[["model", "roc_auc", "accuracy", "purpose"]].round(4))
pd.read_csv(ev.out.checks / "priority_test.csv").round(4)

## 6. Novelty checks

Compare the as-of split against no-purge split and random split, and check if the explanation really describe the model or not.

In [ ]:
protocols, checks = evaluate.run_checks(ev)
display(protocols[["protocol", "model", "roc_auc_mean", "roc_auc_sd", "delta_vs_asof"]].round(4))
pd.json_normalize(checks, sep=".").T

## 7. Record the stage

In [ ]:
manifest = evaluate.finish(ev)

Next: `3_predict.ipynb`.